In [0]:
%run ../Includes/Copy-Datasets

In [0]:
%run Users/sravan.m@anblicks.com/Learning_db/Includes/Copy-Datasets

In [0]:
 %python
 files = dbutils.fs.ls(f"{dataset_bookstore}/customers-json")
 display(files)

In [0]:
%python
files = dbutils.fs.ls("dbfs:/mnt/demo-datasets/bookstore/orders-json-raw/")
display(files)


In [0]:
%python
dbutils.fs.ls(f"{dataset_bookstore}/")

In [0]:
%python
f=dbutils.fs.ls("dbfs:/mnt/demo-datasets/bookstore/books-cdc/")
display(f)


In [0]:
SELECT * FROM json.`dbfs:/mnt/demo-datasets/bookstore/customers-json/export_*.json`

In [0]:
'dbfs:/mnt/demo-datasets/bookstore/orders/'

In [0]:
create table customers
Location '/Workspace/Users/sravan.m@anblicks.com/Learning/Dev/Files/' 
 as 
(SELECT * FROM json.`dbfs:/mnt/demo-datasets/bookstore/customers-json/export_*.json`)


In [0]:
select * from customers

In [0]:
describe extended customers

In [0]:
describe table customers

In [0]:
describe detail customers

In [0]:
describe history customers 

In [0]:
SELECT * FROM csv.`dbfs:/mnt/demo-datasets/bookstore/books-csv` 

In [0]:
CREATE TABLE books_unparsed AS
SELECT * FROM csv.`dbfs:/mnt/demo-datasets/bookstore/books-csv`;

SELECT * FROM books_unparsed;

In [0]:
describe extended books_unparsed

In [0]:
CREATE TEMP VIEW books_tmp_vw
   (book_id STRING, title STRING, author STRING, category STRING, price DOUBLE)
USING CSV
OPTIONS (
  path = "dbfs:/mnt/demo-datasets/bookstore/books-csv/export_*.csv",
  header = "true",
  delimiter = ";"
);

CREATE TABLE books AS
  SELECT * FROM books_tmp_vw;
  
SELECT * FROM books

In [0]:
SELECT * FROM read_files(
  'dbfs:/mnt/demo-datasets/bookstore/books-csv/export_*.csv',
  format => 'csv',
  header => 'true',
  delimiter => ';'
);

In [0]:
CREATE TABLE books_bkp AS
(
   SELECT * FROM read_files(
  'dbfs:/mnt/demo-datasets/bookstore/books-csv/export_*.csv',
  format => 'csv',
  header => 'true',
  delimiter => ';'
)

);

Select * from books_bkp

In [0]:
describe extended books_bkp


In [0]:
CREATE OR REPLACE TABLE orders AS
SELECT * FROM parquet.`${dataset.bookstore}/orders`

In [0]:
SELECT * FROM parquet.`${dataset.bookstore}/orders`

In [0]:
%python
files=dbutils.fs.ls("dbfs:/mnt/demo-datasets/bookstore/orders/")
display(files) 

In [0]:
SELECT * FROM parquet.`dbfs:/mnt/demo-datasets/bookstore/orders/export_*.parquet`

In [0]:
CREATE OR REPLACE TABLE orders AS

(SELECT * FROM parquet.`dbfs:/mnt/demo-datasets/bookstore/orders/export_*.parquet`)

In [0]:
select * from orders

In [0]:
describe detail orders


In [0]:
SELECT * FROM parquet.`dbfs:/mnt/demo-datasets/bookstore/orders-new/export_*.parquet`

In [0]:
insert into orders
(SELECT * FROM parquet.`dbfs:/mnt/demo-datasets/bookstore/orders-new/export_*.parquet`)


In [0]:
describe history orders

In [0]:
describe detail orders

In [0]:
select * from orders

In [0]:
select * from customers


In [0]:
SELECT * FROM json.`dbfs:/mnt/demo-datasets/bookstore/customers-json-new/export_*.json`

In [0]:
CREATE OR REPLACE TEMP VIEW customers_updates AS 
SELECT * FROM json.`dbfs:/mnt/demo-datasets/bookstore/customers-json-new/export_*.json`;

MERGE INTO customers c
USING customers_updates u
ON c.customer_id = u.customer_id
WHEN MATCHED AND c.email IS NULL AND u.email IS NOT NULL THEN
  UPDATE SET email = u.email, updated = u.updated
WHEN NOT MATCHED THEN INSERT *

In [0]:
CREATE TABLE books_bkp AS
(
   SELECT * FROM read_files(
  'dbfs:/mnt/demo-datasets/bookstore/books-csv/export_*.csv',
  format => 'csv',
  header => 'true',
  delimiter => ';'
)

);

In [0]:
SELECT * FROM read_files(
  'dbfs:/mnt/demo-datasets/bookstore/books-csv-new/export_*.csv',
  format => 'csv',
  header => 'true',
  delimiter => ';'
)

In [0]:
CREATE OR REPLACE TEMP VIEW books_updates AS
SELECT * FROM read_files(
  'dbfs:/mnt/demo-datasets/bookstore/books-csv-new/export_*.csv',
  format => 'csv',
  header => 'true',
  delimiter => ';'
)

In [0]:
SELECT * FROM books_updates

In [0]:
select * from books

In [0]:
MERGE INTO books b
USING books_updates u
ON b.book_id = u.book_id AND b.title = u.title
WHEN NOT MATCHED AND u.category = 'Computer Science' THEN 
  INSERT *

In [0]:
select * from books


In [0]:
describe customers

In [0]:
SELECT customer_id, profile:first_name, profile:address:country 
FROM customers

In [0]:
select * from customers

In [0]:
select * from books

In [0]:
select * from orders

In [0]:
select order_id, order_timestamp, customer_id, quantity, total, books:book_id, books:quantity, books:subtotal from orders


In [0]:
SELECT from_json(profile, 'schema') AS profile_struct
  FROM customers;

In [0]:
 SELECT customer_id, from_json(profile, schema_of_json('{"first_name":"Thomas","last_name":"Lane","gender":"Male","address":{"street":"06 Boulevard Victor Hugo","city":"Paris","country":"France"}}')) AS profile_struct
  FROM customers

In [0]:
select * from customers


In [0]:
SELECT customer_id,email, profile:first_name,profile:last_name, profile:gender, profile:address:country, profile:address:street,profile:address:city
FROM customers

In [0]:
CREATE OR REPLACE TEMP VIEW customers_extended AS
(SELECT customer_id,email, profile:first_name,profile:last_name, profile:gender, profile:address:country, profile:address:street,profile:address:city
FROM customers)

In [0]:
describe extended customers_extended

In [0]:
select * from customers_extended

In [0]:
SELECT customer_id,email,explode(profile)as CustomerInfo
FROM customers

In [0]:
SELECT customer_id,email,explode(split(profile, ',')) as CustomerInfo from customers

In [0]:
select *   FROM orders

In [0]:
SELECT order_id, customer_id, quantity, total, order_timestamp,explode(books) AS book 
FROM orders

In [0]:
SELECT *, explode(books) AS book 
  FROM orders

In [0]:
SELECT *
FROM (
  SELECT *, explode(books) AS book 
  FROM orders) o
INNER JOIN books b
ON o.book.book_id = b.book_id;